# 02 — Data Cleaning: Driver Uber & Rapido (India)

Notebook ini mengubah `uber_rapido_driver_dirty.csv` (50.000 baris, 33 kolom) menjadi `drivers_clean.csv` (48.568 baris, 46 kolom). Temuan masalahnya ada di `01_drivers_data_understanding.ipynb`; keputusan per kolom dirangkum di `docs/cleaning_log.md`.

**Cara pakai:** jalankan **Restart and Run All** dari atas ke bawah. Sel bertanda `cek(...)` mencetak ✅ bila hasilnya sama dengan angka yang sudah diverifikasi dan ❌ bila berbeda.

> **Perhatian:** sel penyimpanan di bagian 11 menimpa `drivers_clean.csv`. Kalau ingin membandingkan dengan hasil lama, salin dulu file itu menjadi `drivers_clean_lama.csv`.

**Alur:**
1. Muat data
2. Duplikat identik
3. Placeholder tersembunyi
4. Standardisasi kolom kategori
5. Kolom numerik
6. Kolom boolean
7. Nama, duplikat setelah standardisasi, dan ID
8. Tanggal
9. Tipe data
10. Kolom turunan
11. Validasi dan penyimpanan

## 1. Memuat data

In [1]:
import os

import numpy as np
import pandas as pd

RAW_FILE = "uber_rapido_driver_dirty.csv"
OUT_FILE = "drivers_clean.csv"
BACKUP_FILE = "drivers_clean_lama.csv"
DOCS_DIR = "docs"
os.makedirs(DOCS_DIR, exist_ok=True)


def cek(nama, hasil, harapan):
    """Cetak ✅ bila hasil sama dengan angka yang sudah diverifikasi, ❌ bila berbeda."""
    status = "✅" if hasil == harapan else "❌"
    print(f"{status} {nama}: {hasil} (harapan: {harapan})")

In [2]:
df = pd.read_csv(RAW_FILE)
dtype_asli = df.dtypes.astype(str)     # disimpan untuk laporan perubahan tipe data di bagian 11

cek("baris mentah", len(df), 50000)
cek("kolom mentah", df.shape[1], 33)

✅ baris mentah: 50000 (harapan: 50000)
✅ kolom mentah: 33 (harapan: 33)


## 2. Duplikat identik

In [3]:
n_dup_identik = int(df.duplicated().sum())
df = df.drop_duplicates()

cek("duplikat identik dibuang", n_dup_identik, 951)
print("baris tersisa:", len(df))

✅ duplikat identik dibuang: 951 (harapan: 951)
baris tersisa: 49049


## 3. Placeholder tersembunyi

Tiga lapis: (1) teks placeholder yang pasti kosong diganti `NaN`, (2) spasi di awal dan akhir dibuang dari semua kolom teks, lalu sel yang hanya berisi spasi menjadi `NaN`, (3) `na` dan `unknown` dalam berbagai penulisan huruf diganti `NaN` pada kolom kategori. Bagian (3) dikerjakan pada salinan kerja `dfc`; `df` tetap menyimpan nilai asli untuk perbandingan.

In [4]:
fake_nulls = ["NULL", "N/A", "-", "none", "unknown"]
df = df.replace(fake_nulls, np.nan)

for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].str.strip()

df = df.replace(r"^\s*$", np.nan, regex=True)

In [5]:
dfc = df.copy()

PH = {"na", "unknown"}
cols_ph = ["platform", "vehicle_type", "vehicle_condition", "experience_level",
           "preferred_shift", "reason_for_joining", "languages_known",
           "document_verification", "background_check_status",
           "interview_result", "rejection_reason"]

for c in cols_ph:
    is_ph = dfc[c].str.strip().str.lower().isin(PH)
    dfc[c] = dfc[c].mask(is_ph)

for c in cols_ph:
    sebelum, sesudah = df[c].isna().sum(), dfc[c].isna().sum()
    print(f"{c}: {sebelum} -> {sesudah} (bertambah {sesudah - sebelum})")

platform: 3277 -> 3950 (bertambah 673)
vehicle_type: 3306 -> 4020 (bertambah 714)
vehicle_condition: 11344 -> 12044 (bertambah 700)
experience_level: 3260 -> 3973 (bertambah 713)
preferred_shift: 12034 -> 12813 (bertambah 779)
reason_for_joining: 16702 -> 17426 (bertambah 724)
languages_known: 3197 -> 3959 (bertambah 762)
document_verification: 14597 -> 15363 (bertambah 766)
background_check_status: 14332 -> 15039 (bertambah 707)
interview_result: 5747 -> 6457 (bertambah 710)
rejection_reason: 39408 -> 40118 (bertambah 710)


## 4. Standardisasi kolom kategori

### 4.1 Jenis kendaraan, kondisi kendaraan, pengalaman, hasil interview

In [6]:
maps = {
    "vehicle_type": {"mini": "Mini", "sedan": "Sedan", "bike": "Bike",
                     "suv": "SUV", "auto": "Auto"},
    "vehicle_condition": {"avg": "Average", "average": "Average", "good": "Good",
                          "excellent": "Excellent", "poor": "Poor"},
    "experience_level": {"fresher": "Fresher", "0-1 yr": "0-1 yr", "1-3 yrs": "1-3 yrs",
                         "3-5 yrs": "3-5 yrs", "5+ yrs": "5+ yrs"},
    "interview_result": {"selected": "Selected", "rejected": "Rejected",
                         "on hold": "On Hold", "waitlist": "Waitlist"},
}

for col, mp in maps.items():
    s = dfc[col].str.strip().str.lower()
    dfc[col] = s.replace(mp)
    tidak_terpetakan = set(dfc[col].dropna()) - set(mp.values())
    print(col, "| tidak terpetakan:", tidak_terpetakan)      # harus set()

vehicle_type | tidak terpetakan: set()
vehicle_condition | tidak terpetakan: set()
experience_level | tidak terpetakan: set()
interview_result | tidak terpetakan: set()


### 4.2 Kota dan platform

In [7]:
FAKE = {"", "null", "n/a", "na", "-", "none", "unknown", "nan"}


def norm(s):
    s = s.str.strip().str.lower()
    return s.mask(s.isin(FAKE))


city_typo = {
    "mumabi": "mumbai",
    "hydrabad": "hyderabad",
    "bangalroe": "bangalore",
    "dlhi": "delhi",
    "chenna": "chennai",
}

platform_map = {
    "uber": "Uber",
    "rapido": "Rapido",
    "uber & rapido": "Uber & Rapido",
    "ola": "Ola",
}

dfc["city"] = norm(dfc["city"]).replace(city_typo).str.title()
dfc["platform"] = norm(dfc["platform"]).replace(platform_map)

In [8]:
kota_valid = {"Ahmedabad", "Bangalore", "Chennai", "Delhi", "Hyderabad",
              "Jaipur", "Kolkata", "Lucknow", "Mumbai", "Pune"}

cek("jumlah kota unik", dfc["city"].nunique(), 10)
cek("kota di luar daftar valid", set(dfc["city"].dropna()) - kota_valid, set())
cek("kota kosong bertambah", int(dfc["city"].isna().sum()), int(df["city"].isna().sum()))
dfc["platform"].value_counts(dropna=False)

✅ jumlah kota unik: 10 (harapan: 10)
✅ kota di luar daftar valid: set() (harapan: set())
✅ kota kosong bertambah: 0 (harapan: 0)


platform
Uber             18079
Rapido           17910
Ola               4556
Uber & Rapido     4554
NaN               3950
Name: count, dtype: int64

### 4.3 Bahasa

`languages_known` berisi beberapa bahasa dalam satu sel dan hanya dibersihkan dari placeholder. Kolom bantu `n_languages` (jumlah bahasa) dihitung di sini; ia tidak masuk ke data akhir, tetapi ikut terhitung saat memilih baris terbaik di bagian 7, sama seperti pada versi sebelumnya.

In [9]:
lang_list = dfc["languages_known"].str.split(",").apply(
    lambda x: sorted(i.strip() for i in x) if isinstance(x, list) else np.nan)
dfc["n_languages"] = lang_list.apply(lambda x: len(x) if isinstance(x, list) else np.nan)

langs = dfc["languages_known"].dropna().str.split(",").explode().str.strip()
cek("jumlah bahasa unik", langs.nunique(), 8)

✅ jumlah bahasa unik: 8 (harapan: 8)


### 4.4 Shift, verifikasi dokumen, background check, alasan penolakan

In [10]:
dfc["document_verification"] = (dfc["document_verification"].str.strip().str.lower()
    .replace({"pass": "Pass", "fail": "Fail", "pending": "Pending"}))

dfc["background_check_status"] = (dfc["background_check_status"].str.strip().str.lower()
    .replace({"clear": "Clear", "pending": "Pending", "flagged": "Flagged"}))

dfc["preferred_shift"] = (dfc["preferred_shift"].str.strip().str.lower()
    .replace({"day": "Day", "night": "Night", "both": "Both"}))

dfc["rejection_reason"] = dfc["rejection_reason"].str.strip().str.lower().str.capitalize()

In [11]:
for c in ["preferred_shift", "document_verification", "background_check_status",
          "reason_for_joining", "rejection_reason"]:
    print(c, "| nilai unik:", dfc[c].nunique())

# alasan penolakan tidak hanya terisi untuk yang ditolak; hubungan ini dicatat, bukan diubah
pd.crosstab(dfc["interview_result"].fillna("MISSING"), dfc["rejection_reason"].notna())

preferred_shift | nilai unik: 3
document_verification | nilai unik: 3
background_check_status | nilai unik: 3
reason_for_joining | nilai unik: 7
rejection_reason | nilai unik: 7


rejection_reason,False,True
interview_result,,
MISSING,5313,1144
On Hold,11094,480
Rejected,6375,6633
Selected,16318,449
Waitlist,1018,225


## 5. Kolom numerik

Aturan umum: tanda minus yang jelas salah ketik dibalik dengan `abs()` dan ditandai di kolom `*_sign_fixed`; nilai penanda (999, 99999, 9999999) dan nilai di luar rentang yang masuk akal dikosongkan.

### 5.1 Umur

In [12]:
age_num = dfc["age"].str.extract(r"(-?\d+\.?\d*)")[0].astype(float)
dfc["age_sign_fixed"] = age_num < 0
dfc["age_clean"] = age_num.mask(age_num == 999).abs()

print(dfc["age_clean"].describe())
cek("umur kosong", int(dfc["age_clean"].isna().sum()), int(dfc["age"].isna().sum() + (age_num == 999).sum()))

count    44166.000000
mean        37.553435
std         10.362768
min         20.000000
25%         29.000000
50%         38.000000
75%         47.000000
max         55.000000
Name: age_clean, dtype: float64
✅ umur kosong: 4883 (harapan: 4883)


### 5.2 Rating dan skor interview

In [13]:
score_cols = ["communication_score_10", "route_knowledge_score_10", "customer_handling_score_10",
              "driving_test_score_10", "overall_interview_score_10"]

# rating: "unrated" otomatis menjadi NaN lewat errors="coerce"; di luar 0-5 dikosongkan
r = pd.to_numeric(dfc["overall_driver_rating"], errors="coerce")
dfc["rating_clean"] = r.where((r >= 0) & (r <= 5))

# skor: di luar 0-10 dikosongkan
for c in score_cols:
    s = dfc[c]
    dfc[c + "_clean"] = s.where((s >= 0) & (s <= 10))

for c in ["rating_clean"] + [c + "_clean" for c in score_cols]:
    print(c, dfc[c].notna().sum(), dfc[c].min(), dfc[c].max())

rating_clean 39229 2.5 5.0
communication_score_10_clean 39721 3.0 10.0
route_knowledge_score_10_clean 39767 3.0 10.0
customer_handling_score_10_clean 39736 3.0 10.0
driving_test_score_10_clean 39750 4.0 10.0
overall_interview_score_10_clean 39865 3.4 9.8


### 5.3 Trips, rate, complaints, compliments

In [14]:
# trips: negatif dimutlakkan dan ditandai, 99999 dikosongkan
t = dfc["total_trips_completed"]
dfc["trips_sign_fixed"] = t < 0
dfc["trips_clean"] = t.mask(t == 99999).abs()

# rate: negatif dan di atas 100 dikosongkan
for c, new in [("trip_completion_rate_%", "trip_completion_rate_clean"),
               ("cancellation_rate_%", "cancellation_rate_clean")]:
    s = dfc[c]
    dfc[new] = s.where(s.between(0, 100))

# complaints dan compliments: 99999 dikosongkan
for c in ["total_complaints", "total_compliments"]:
    s = dfc[c]
    dfc[c + "_clean"] = s.mask(s == 99999)

for c in ["trips_clean", "trip_completion_rate_clean", "cancellation_rate_clean",
          "total_complaints_clean", "total_compliments_clean"]:
    print(c, dfc[c].notna().sum(), dfc[c].min(), dfc[c].max())

trips_clean 43547 10.0 8000.0
trip_completion_rate_clean 39782 60.0 100.0
cancellation_rate_clean 39570 0.0 35.0
total_complaints_clean 41120 0.0 50.0
total_compliments_clean 41117 0.0 100.0


### 5.4 Penghasilan harian

In [15]:
# format campuran ("Rs.1200", "₹1200", "1200/-", "-1200"): ambil angkanya saja
earn = dfc["avg_daily_earnings_INR"].str.extract(r"(-?\d+\.?\d*)")[0].astype(float)
dfc["earn_sign_fixed"] = earn < 0
dfc["earn_clean"] = earn.mask(earn == 9999999).abs()

cek("angka penghasilan hilang saat ekstraksi",
    int(earn.notna().sum()), int(dfc["avg_daily_earnings_INR"].notna().sum()))
print(dfc["earn_clean"].describe())

✅ angka penghasilan hilang saat ekstraksi: 46586 (harapan: 46586)
count    46092.000000
mean      1444.681138
std        605.578041
min        400.000000
25%        922.000000
50%       1439.000000
75%       1967.000000
max       2500.000000
Name: earn_clean, dtype: float64


## 6. Kolom boolean

In [16]:
bool_map = {"yes": True, "true": True, "1": True, "y": True,
            "no": False, "false": False, "0": False, "n": False}

s = dfc["is_currently_active"].str.strip().str.lower()
dfc["is_active_clean"] = s.map(bool_map).astype("boolean")

cek("nilai tak dikenal", list(s[~s.isin(bool_map)].unique()), [])
print(dfc["is_active_clean"].value_counts(dropna=False))

✅ nilai tak dikenal: [] (harapan: [])
is_active_clean
True     24572
False    24477
Name: count, dtype: Int64


## 7. Nama, duplikat setelah standardisasi, dan ID

Setelah teks dirapikan, sebagian baris yang tadinya tampak berbeda ternyata kembar. Dua baris dianggap kembar bila delapan kolom kunci sama; dari tiap pasangan dipertahankan baris dengan nilai kosong paling sedikit.

In [17]:
name = (dfc["driver_name"].str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.title())
dfc["driver_name_clean"] = name
print("nama unik:", dfc["driver_name"].nunique(), "->", name.nunique())

nama unik: 2360 -> 1150


In [18]:
key = ["driver_id", "driver_name_clean", "city", "platform",
       "age_clean", "vehicle_type", "trips_clean", "earn_clean"]

cek("baris yang akan dibuang", int(dfc.duplicated(subset=key).sum()), 481)

dfc["n_missing"] = dfc.isna().sum(axis=1)
dfc = (dfc.sort_values("n_missing", kind="stable")
          .drop_duplicates(subset=key, keep="first")
          .sort_index()
          .drop(columns="n_missing"))

cek("baris setelah duplikat dibuang", len(dfc), 48568)

✅ baris yang akan dibuang: 481 (harapan: 481)
✅ baris setelah duplikat dibuang: 48568 (harapan: 48568)


`driver_id` tidak unik dan hampir semua ID yang bertabrakan dimiliki orang berbeda, jadi ID tidak diubah. Setiap baris diberi `record_id` sebagai kunci unik, dan baris yang ID-nya bertabrakan diberi penanda `id_collision`.

In [19]:
dfc = dfc.reset_index(drop=True)
dfc["record_id"] = dfc.index + 1
dfc["id_collision"] = dfc["driver_id"].notna() & dfc.duplicated("driver_id", keep=False)

cek("record_id unik", bool(dfc["record_id"].is_unique), True)
cek("baris dengan ID bertabrakan", int(dfc["id_collision"].sum()), 2651)

✅ record_id unik: True (harapan: True)
✅ baris dengan ID bertabrakan: 2651 (harapan: 2651)


In [20]:
tabrakan = dfc[dfc["id_collision"]]
g = tabrakan.groupby("driver_id").agg(baris=("record_id", "size"),
                                      nama_unik=("driver_name_clean", "nunique"))
cek("ID yang bertabrakan", len(g), 1302)
print("nama beda (orang berbeda):", int((g["nama_unik"] > 1).sum()))
print(g["baris"].value_counts())

✅ ID yang bertabrakan: 1302 (harapan: 1302)
nama beda (orang berbeda): 1301
baris
2    1255
3      47
Name: count, dtype: int64


## 8. Tanggal

Ada tujuh format tanggal. Bentuk setiap nilai dikenali dengan pola (angka menjadi `9`, huruf menjadi `A`), lalu di-parse sesuai formatnya. Urutan hari-bulan pada format angka ditentukan dari rentang nilainya di `01_drivers_data_understanding.ipynb`. Tanggal yang hanya berisi bulan dan tahun diberi penanda `*_month_only` dan tidak dipakai untuk menghitung lama onboarding.

In [21]:
FORMATS = {
    "9999-99-99A99:99:99A": "iso",
    "99 A 9999": "%d %B %Y",
    "A 9999": "%B %Y",
    "99-99-9999": "%d-%m-%Y",
    "99.99.9999": "%d.%m.%Y",
    "99/99/99": "%m/%d/%y",
    "9999/99/99": "%Y/%m/%d",
}


def parse_mixed(col):
    s = col.dropna().astype(str).str.strip()
    shape = (s.str.replace(r"\d", "9", regex=True)
               .str.replace(r"[A-Za-z]+", "A", regex=True))

    unknown = shape[~shape.isin(FORMATS)].value_counts()
    if len(unknown):
        print("bentuk tak dikenal:", unknown.to_dict())

    pieces = []
    for sh, f in FORMATS.items():
        m = s[shape == sh]
        if m.empty:
            continue
        if f == "iso":
            pieces.append(pd.to_datetime(m, utc=True).dt.tz_localize(None))
        else:
            pieces.append(pd.to_datetime(m, format=f))

    out = pd.concat(pieces).reindex(col.index)
    month_only = (shape == "A 9999").reindex(col.index, fill_value=False)
    return out, month_only


dfc["onboarding_date_clean"], dfc["onb_month_only"] = parse_mixed(dfc["onboarding_date"])
dfc["interview_date_clean"], dfc["int_month_only"] = parse_mixed(dfc["interview_date"])

cek("interview_date gagal di-parse",
    int(dfc["interview_date"].notna().sum() - dfc["interview_date_clean"].notna().sum()), 0)
cek("onboarding_date gagal di-parse",
    int(dfc["onboarding_date"].notna().sum() - dfc["onboarding_date_clean"].notna().sum()), 0)
print("rentang onboarding:", dfc["onboarding_date_clean"].min(), "sampai", dfc["onboarding_date_clean"].max())

✅ interview_date gagal di-parse: 0 (harapan: 0)
✅ onboarding_date gagal di-parse: 0 (harapan: 0)
rentang onboarding: 2021-01-01 00:00:00 sampai 2024-06-14 00:00:00


In [22]:
# lama onboarding hanya dihitung bila kedua tanggal lengkap (bukan bulan-saja)
full = ~dfc["onb_month_only"] & ~dfc["int_month_only"]
dfc["days_to_onboard"] = ((dfc["onboarding_date_clean"] - dfc["interview_date_clean"])
                          .dt.days.where(full))

cek("baris dengan days_to_onboard", int(dfc["days_to_onboard"].notna().sum()), 12163)
cek("onboarding sebelum interview", int((dfc["days_to_onboard"] < 0).sum()), 0)
print(dfc["days_to_onboard"].describe())

✅ baris dengan days_to_onboard: 12163 (harapan: 12163)
✅ onboarding sebelum interview: 0 (harapan: 0)
count    12163.000000
mean        30.429993
std         17.275230
min          1.000000
25%         15.000000
50%         30.000000
75%         45.500000
max         60.000000
Name: days_to_onboard, dtype: float64


## 9. Tipe data

Kolom bilangan bulat diubah ke `Int64` (mendukung nilai kosong dan tidak tertulis `.0` di CSV), dan kolom kategori ke `category`. Kolom `peak_hour_availability` dijadikan boolean `is_peak_available`.

In [23]:
int_cols = ["age_clean", "trips_clean", "total_complaints_clean",
            "total_compliments_clean", "days_to_onboard"]

# pastikan memang bilangan bulat sebelum dikonversi
for c in int_cols:
    cek(f"{c} semua bilangan bulat", bool((dfc[c].dropna() % 1 == 0).all()), True)

✅ age_clean semua bilangan bulat: True (harapan: True)
✅ trips_clean semua bilangan bulat: True (harapan: True)
✅ total_complaints_clean semua bilangan bulat: True (harapan: True)
✅ total_compliments_clean semua bilangan bulat: True (harapan: True)
✅ days_to_onboard semua bilangan bulat: True (harapan: True)


In [24]:
for c in int_cols:
    dfc[c] = dfc[c].astype("Int64")

cat_cols = ["city", "platform", "vehicle_type", "vehicle_condition", "experience_level",
            "preferred_shift", "reason_for_joining", "document_verification",
            "background_check_status", "interview_result", "rejection_reason"]
for c in cat_cols:
    dfc[c] = dfc[c].astype("category")

s = dfc["peak_hour_availability"].str.strip().str.lower()
dfc["is_peak_available"] = s.map(bool_map).astype("boolean")

cek("nilai peak tak dikenal", list(s[~s.isin(bool_map)].unique()), [])
cek("is_peak_available kosong", int(dfc["is_peak_available"].isna().sum()), 0)
print(dfc[int_cols + cat_cols].dtypes.to_string())

✅ nilai peak tak dikenal: [] (harapan: [])
✅ is_peak_available kosong: 0 (harapan: 0)
age_clean                     Int64
trips_clean                   Int64
total_complaints_clean        Int64
total_compliments_clean       Int64
days_to_onboard               Int64
city                       category
platform                   category
vehicle_type               category
vehicle_condition          category
experience_level           category
preferred_shift            category
reason_for_joining         category
document_verification      category
background_check_status    category
interview_result           category
rejection_reason           category


## 10. Kolom turunan

Tujuh kolom kategori dibuat untuk memudahkan analisis dan dashboard. Batas kelompoknya: umur 20-29, 30-39, 40-49, 50-55; penghasilan Low (di bawah 1.000), Medium (1.000 sampai di bawah 2.000), High (2.000 ke atas); jumlah trip Low (di bawah 2.000), Medium (2.000 sampai di bawah 5.000), High (5.000 ke atas); rating Low (di bawah 3,5), Medium (3,5 sampai di bawah 4,5), High (4,5 ke atas); lama onboarding Fast (1-15 hari), Normal (16-45), Slow (46-60).

`funnel_stage` menggabungkan hasil interview dengan ada-tidaknya tanggal onboarding.

In [25]:
dfc["age_group"] = pd.cut(dfc["age_clean"].astype(float), bins=[19, 29, 39, 49, 55],
                          labels=["20-29", "30-39", "40-49", "50-55"])
dfc["earnings_category"] = pd.cut(dfc["earn_clean"], bins=[0, 1000, 2000, np.inf],
                                  labels=["Low", "Medium", "High"], right=False)
dfc["trip_volume_category"] = pd.cut(dfc["trips_clean"].astype(float), bins=[0, 2000, 5000, np.inf],
                                     labels=["Low", "Medium", "High"], right=False)
dfc["rating_tier"] = pd.cut(dfc["rating_clean"], bins=[0, 3.5, 4.5, 5.1],
                            labels=["Low", "Medium", "High"], right=False)
dfc["onboarding_speed"] = pd.cut(dfc["days_to_onboard"].astype(float), bins=[0, 15, 45, 60],
                                 labels=["Fast", "Normal", "Slow"])
dfc["is_multi_platform"] = dfc["platform"].astype("string").eq("Uber & Rapido").astype("boolean")

ir = dfc["interview_result"].astype("object")
has_onb = dfc["onboarding_date_clean"].notna()
dfc["funnel_stage"] = pd.Series(np.select(
    [ir.eq("Rejected"), ir.eq("Selected") & has_onb, ir.eq("Selected") & ~has_onb, ir.isin(["On Hold", "Waitlist"])],
    ["Rejected", "Selected - Onboarded", "Selected - No onboarding date", "Pending (On Hold/Waitlist)"],
    default="Unknown"), index=dfc.index).astype("category")

dfc["funnel_stage"].value_counts(dropna=False)

funnel_stage
Selected - Onboarded             13650
Rejected                         12882
Pending (On Hold/Waitlist)       12696
Unknown                           6379
Selected - No onboarding date     2961
Name: count, dtype: int64

## 11. Validasi dan penyimpanan

Data akhir berisi 46 kolom: **31 kolom hasil pembersihan** (nama kolom asli yang dibersihkan atau diganti namanya, misalnya `age` menjadi `age_clean`) dan **15 kolom turunan** baru. `phone` dan `email` tidak dibawa.

In [26]:
final_cols = [
    "record_id", "driver_id", "id_collision", "driver_name_clean",
    "age_clean", "age_sign_fixed", "city", "platform", "vehicle_type",
    "vehicle_condition", "experience_level", "preferred_shift",
    "reason_for_joining", "languages_known",
    "document_verification", "background_check_status",
    "interview_result", "rejection_reason",
    "interview_date_clean", "onboarding_date_clean",
    "int_month_only", "onb_month_only", "days_to_onboard",
    "rating_clean", "communication_score_10_clean", "route_knowledge_score_10_clean",
    "customer_handling_score_10_clean", "driving_test_score_10_clean",
    "overall_interview_score_10_clean",
    "trips_clean", "trips_sign_fixed", "trip_completion_rate_clean",
    "cancellation_rate_clean", "total_complaints_clean", "total_compliments_clean",
    "earn_clean", "earn_sign_fixed", "is_active_clean",
    "is_peak_available",
    "age_group", "earnings_category", "trip_volume_category",
    "rating_tier", "onboarding_speed", "is_multi_platform", "funnel_stage",
]

kolom_turunan = [
    "record_id", "id_collision", "age_sign_fixed", "trips_sign_fixed", "earn_sign_fixed",
    "int_month_only", "onb_month_only", "days_to_onboard",
    "age_group", "earnings_category", "trip_volume_category",
    "rating_tier", "onboarding_speed", "is_multi_platform", "funnel_stage",
]

cek("kolom akhir", len(final_cols), 46)
cek("kolom unik", len(set(final_cols)), 46)
cek("kolom yang belum ada di dfc", [c for c in final_cols if c not in dfc.columns], [])
cek("kolom turunan", len(kolom_turunan), 15)
cek("kolom hasil pembersihan", len(final_cols) - len(kolom_turunan), 31)

✅ kolom akhir: 46 (harapan: 46)
✅ kolom unik: 46 (harapan: 46)
✅ kolom yang belum ada di dfc: [] (harapan: [])
✅ kolom turunan: 15 (harapan: 15)
✅ kolom hasil pembersihan: 31 (harapan: 31)


In [27]:
dfc[final_cols].to_csv(OUT_FILE, index=False)

cek_csv = pd.read_csv(OUT_FILE, dtype=str)
cek("shape drivers_clean.csv", cek_csv.shape, (48568, 46))
print("contoh age_clean :", cek_csv["age_clean"].dropna().head(3).tolist())     # tanpa .0
print("contoh trips_clean:", cek_csv["trips_clean"].dropna().head(3).tolist())  # tanpa .0

✅ shape drivers_clean.csv: (48568, 46) (harapan: (48568, 46))
contoh age_clean : ['45', '53', '46']
contoh trips_clean: ['2752', '5073', '262']


### Perbandingan dengan hasil sebelumnya

In [28]:
if os.path.exists(BACKUP_FILE):
    lama = pd.read_csv(BACKUP_FILE, dtype=str)
    baru = pd.read_csv(OUT_FILE, dtype=str)
    print("shape lama:", lama.shape, "| shape baru:", baru.shape)
    print("nama dan urutan kolom sama:", lama.columns.tolist() == baru.columns.tolist())
    print("isi sama persis:", lama.equals(baru))
    if lama.shape == baru.shape and lama.columns.tolist() == baru.columns.tolist() and not lama.equals(baru):
        beda = (lama.fillna("") != baru.fillna("")).sum()
        print("kolom yang berbeda:")
        print(beda[beda > 0])
else:
    print(f"{BACKUP_FILE} tidak ditemukan; lewati perbandingan.")

shape lama: (48568, 46) | shape baru: (48568, 46)
nama dan urutan kolom sama: True
isi sama persis: True


### Laporan kualitas data

Tiga tabel kecil disimpan ke folder `docs/` sebagai bahan `cleaning_log.md`: aturan validitas pada data mentah, perubahan tipe data, dan jumlah outlier (metode IQR) pada data bersih. Outlier hanya dilaporkan, tidak dibuang.

In [29]:
src = dfc      # kolom mentah masih ada di dfc


def num(col):
    s = src[col]
    if pd.api.types.is_numeric_dtype(s):
        return s.astype(float)
    return s.astype(str).str.extract(r"(-?\d+\.?\d*)")[0].astype(float)


rows = []


def add(aturan, series, valid):
    s = series.dropna()
    bad = int((~valid(s)).sum())
    rows.append((aturan, len(s), bad, round(bad / len(s) * 100, 2), "✅" if bad == 0 else "❌"))


add("age antara 18 dan 65", num("age"), lambda s: s.between(18, 65))
add("rating antara 0 dan 5", num("overall_driver_rating"), lambda s: s.between(0, 5))
add("5 skor interview antara 0 dan 10", pd.concat([num(c) for c in score_cols]), lambda s: s.between(0, 10))
add("trip_completion_rate antara 0 dan 100", num("trip_completion_rate_%"), lambda s: s.between(0, 100))
add("cancellation_rate antara 0 dan 100", num("cancellation_rate_%"), lambda s: s.between(0, 100))
add("total_trips >= 0 dan bukan 99999", num("total_trips_completed"), lambda s: (s >= 0) & (s != 99999))
add("total_complaints >= 0 dan bukan 99999", num("total_complaints"), lambda s: (s >= 0) & (s != 99999))
add("total_compliments >= 0 dan bukan 99999", num("total_compliments"), lambda s: (s >= 0) & (s != 99999))
add("earnings > 0 dan bukan 9999999", num("avg_daily_earnings_INR"), lambda s: (s > 0) & (s != 9999999))

ok_bool = {"yes", "no", "true", "false", "1", "0", "y", "n"}
add("is_currently_active berupa nilai boolean yang dikenal",
    src["is_currently_active"].astype(str).str.strip().str.lower(), lambda s: s.isin(ok_bool))

today = pd.Timestamp.today().normalize()
add("tanggal interview tidak di masa depan", src["interview_date_clean"], lambda s: s <= today)
add("tanggal onboarding tidak di masa depan", src["onboarding_date_clean"], lambda s: s <= today)
gap = (src["onboarding_date_clean"] - src["interview_date_clean"]).dt.days.where(full)
add("onboarding tidak sebelum interview", gap, lambda s: s >= 0)

ids = src["driver_id"].dropna()
bad = int(ids.duplicated(keep=False).sum())
rows.append(("driver_id harus unik", len(ids), bad, round(bad / len(ids) * 100, 2), "✅" if bad == 0 else "❌"))

validity = pd.DataFrame(rows, columns=["Aturan validitas", "n terisi", "n tidak valid", "% tidak valid", "Status"])
validity

,Aturan validitas,n terisi,n tidak valid,% tidak valid,Status
0,age antara 18 dan 65,45175,3852,8.53,❌
1,rating antara 0 dan 5,43758,4912,11.23,❌
2,5 skor interview antara 0 dan 10,220966,24069,10.89,❌
3,trip_completion_rate antara 0 dan 100,44245,4869,11.00,❌
4,cancellation_rate antara 0 dan 100,44188,4980,11.27,❌
5,total_trips >= 0 dan bukan 99999,44624,3836,8.60,❌
6,total_complaints >= 0 dan bukan 99999,44647,3933,8.81,❌
7,total_compliments >= 0 dan bukan 99999,44698,3971,8.88,❌
8,earnings > 0 dan bukan 9999999,46128,1450,3.14,❌
9,is_currently_active berupa nilai boolean yang ...,48568,0,0.00,✅


In [30]:
clean_name = {
    "driver_name": "driver_name_clean", "age": "age_clean",
    "overall_driver_rating": "rating_clean", "total_trips_completed": "trips_clean",
    "trip_completion_rate_%": "trip_completion_rate_clean",
    "cancellation_rate_%": "cancellation_rate_clean",
    "avg_daily_earnings_INR": "earn_clean",
    "total_complaints": "total_complaints_clean", "total_compliments": "total_compliments_clean",
    "communication_score_10": "communication_score_10_clean",
    "route_knowledge_score_10": "route_knowledge_score_10_clean",
    "customer_handling_score_10": "customer_handling_score_10_clean",
    "driving_test_score_10": "driving_test_score_10_clean",
    "overall_interview_score_10": "overall_interview_score_10_clean",
    "interview_date": "interview_date_clean", "onboarding_date": "onboarding_date_clean",
    "is_currently_active": "is_active_clean",
    "peak_hour_availability": "is_peak_available",
}

rows = []
for c in df.columns:
    new = clean_name.get(c, c)          # kolom yang tidak ada di kamus dianggap namanya tetap
    if new in dfc.columns and new in final_cols:
        baru = str(dfc[new].dtype)
    else:
        new, baru = "—", "tidak dipakai"
    rows.append((c, dtype_asli[c], new, baru))

tipe_all = pd.DataFrame(rows, columns=["kolom_asli", "tipe_asli", "kolom_bersih", "tipe_baru"])
ubah = tipe_all[(tipe_all["tipe_asli"] != tipe_all["tipe_baru"]) & (tipe_all["tipe_baru"] != "tidak dipakai")]

cek("kolom mentah dipetakan", len(tipe_all), 33)
cek("kolom mentah tidak dibawa (phone, email)", tipe_all.loc[tipe_all["tipe_baru"] == "tidak dipakai", "kolom_asli"].tolist(), ["phone", "email"])
print("kolom yang tipenya berubah:", len(ubah))
ubah

✅ kolom mentah dipetakan: 33 (harapan: 33)
✅ kolom mentah tidak dibawa (phone, email): ['phone', 'email'] (harapan: ['phone', 'email'])
kolom yang tipenya berubah: 21


,kolom_asli,tipe_asli,kolom_bersih,tipe_baru
2,age,str,age_clean,Int64
3,city,str,city,category
6,platform,str,platform,category
7,vehicle_type,str,vehicle_type,category
8,vehicle_condition,str,vehicle_condition,category
9,experience_level,str,experience_level,category
10,preferred_shift,str,preferred_shift,category
11,reason_for_joining,str,reason_for_joining,category
13,overall_driver_rating,str,rating_clean,float64
14,total_trips_completed,float64,trips_clean,Int64


In [31]:
num_cols = ["age_clean", "trips_clean", "earn_clean", "rating_clean",
            "total_complaints_clean", "total_compliments_clean",
            "trip_completion_rate_clean", "cancellation_rate_clean"]
out = {}
for c in num_cols:
    s = dfc[c].dropna().astype(float)
    q1, q3 = s.quantile([.25, .75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int(((s < lo) | (s > hi)).sum())
    out[c] = {"lower_bound": lo, "upper_bound": hi, "n_outlier": n_out, "persen": n_out / len(s) * 100}

outlier = pd.DataFrame(out).round(2)
outlier

,age_clean,trips_clean,earn_clean,rating_clean,total_complaints_clean,total_compliments_clean,trip_completion_rate_clean,cancellation_rate_clean
lower_bound,2.0,-4017.5,-645.5,1.15,-27.0,-52.5,40.9,-17.45
upper_bound,74.0,11978.5,3534.5,6.35,77.0,151.5,119.3,52.55
n_outlier,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.00
persen,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.00


In [32]:
validity.to_csv(os.path.join(DOCS_DIR, "validity_check.csv"), index=False)
ubah.to_csv(os.path.join(DOCS_DIR, "tipe_data_berubah.csv"), index=False)
outlier.to_csv(os.path.join(DOCS_DIR, "outlier_check.csv"))
print("laporan tersimpan di folder", DOCS_DIR)

laporan tersimpan di folder docs
